In [ ]:
%pip install azureml-widgets -q
%pip install azureml-train-automl-runtime==1.57.0 -q
%pip install --upgrade azureml-sdk[notebooks,automl] -q

In [1]:
from azureml.core import Workspace, Experiment
session_id = "267614"
subscription_id= "a24a24d5-8d87-4c8a-99b6-91ed2d2df51f"
resource_group= f"aml-quickstarts-{session_id}"
workspace_name= f"quick-starts-ws-{session_id}"

#ws = Workspace.from_config()
ws = Workspace.get(name=workspace_name, subscription_id=subscription_id, resource_group=resource_group)
exp = Experiment(workspace=ws, name="udacity-project")

print('Workspace name: ' + ws.name, 
      'Azure region: ' + ws.location, 
      'Subscription id: ' + ws.subscription_id, 
      'Resource group: ' + ws.resource_group, sep = '\n')

run = exp.start_logging()

StatementMeta(144621e8-07d0-4231-8e1f-8c74d0028a6b, 0, 6, Finished, Available, Finished)

Performing interactive authentication. Please follow the instructions on the terminal.
To sign in, use a web browser to open the page https://microsoft.com/devicelogin and enter the code ED59VDZKF to authenticate.
Interactive authentication successfully completed.
Workspace name: quick-starts-ws-267614
Azure region: southcentralus
Subscription id: a24a24d5-8d87-4c8a-99b6-91ed2d2df51f
Resource group: aml-quickstarts-267614


In [2]:
from azureml.core.compute import ComputeTarget, AmlCompute
cluster_name = "project-1"
vm_size="Standard_D2_V2"
compute_config=AmlCompute.provisioning_configuration(vm_size=vm_size,max_nodes=4)
# TODO: Create compute cluster
# Use vm_size = "Standard_D2_V2" in your provisioning configuration.
# max_nodes should be no greater than 4.
### YOUR CODE HERE ###
from azureml.exceptions import ComputeTargetException
try:
    compute_target=ComputeTarget(workspace=ws,name=cluster_name)
    print('Found existing compute cluster:',cluster_name)
except ComputeTargetException:
    compute_target=ComputeTarget.create(workspace=ws,name=cluster_name,provisioning_configuration=compute_config)
    compute_target.wait_for_completion(show_output=True)


StatementMeta(144621e8-07d0-4231-8e1f-8c74d0028a6b, 0, 7, Finished, Available, Finished)

InProgress..
SucceededProvisioning operation finished, operation "Succeeded"
Succeeded
AmlCompute wait for completion finished

Minimum number of nodes requested have been provisioned


In [4]:
from azureml.widgets import RunDetails
from azureml.train.sklearn import SKLearn
from azureml.train.hyperdrive.run import PrimaryMetricGoal
from azureml.train.hyperdrive.policy import BanditPolicy
from azureml.train.hyperdrive.sampling import RandomParameterSampling
from azureml.train.hyperdrive.runconfig import HyperDriveConfig
from azureml.train.hyperdrive.parameter_expressions import choice, uniform
from azureml.core import Environment, ScriptRunConfig
import os

# Specify parameter sampler
ps = RandomParameterSampling(parameter_space={'--C':uniform(0.5,1.5),'--max_iter':choice(16,32,64,128)})

# Specify a Policy
policy = BanditPolicy(slack_factor=0.2)

if "training" not in os.listdir():
    os.mkdir("./training")

# Setup environment for your training run
sklearn_env = Environment.from_conda_specification(
    name='sklearn-env', file_path=f'Users/odl_user_{session_id}/conda_dependencies.yml')

# Create a ScriptRunConfig Object to specify the configuration details of your training job
src = ScriptRunConfig(
    source_directory='.',
    script=f'Users/odl_user_{session_id}/train.py',
    environment=sklearn_env,compute_target=compute_target)

# Create a HyperDriveConfig using the src object, hyperparameter sampler, and policy.
hyperdrive_config = HyperDriveConfig(
    hyperparameter_sampling=ps,
    primary_metric_goal=PrimaryMetricGoal.MAXIMIZE,
    primary_metric_name='Accuracy',
    policy=policy,
    run_config=src,
    max_total_runs=15,
    max_concurrent_runs=4)

StatementMeta(144621e8-07d0-4231-8e1f-8c74d0028a6b, 0, 9, Finished, Available, Finished)

In [6]:
# Submit your hyperdrive run to the experiment and show run details with the widget.

### YOUR CODE HERE ###
exp_run=exp.submit(hyperdrive_config)
RunDetails(exp_run).show()
exp_run.wait_for_completion(show_output=True)

StatementMeta(144621e8-07d0-4231-8e1f-8c74d0028a6b, 0, 11, Finished, Available, Finished)

Failed to load image Python extension: libc10_cuda.so: cannot open shared object file: No such file or directory


Initializing logging file for interpret-community


_HyperDriveWidget(widget_settings={'childWidgetDisplay': 'popup', 'send_telemetry': False, 'log_level': 'INFO'…

RunId: HD_d0709859-5831-49d2-b644-561bc8eb1e4d
Web View: https://ml.azure.com/runs/HD_d0709859-5831-49d2-b644-561bc8eb1e4d?wsid=/subscriptions/a24a24d5-8d87-4c8a-99b6-91ed2d2df51f/resourcegroups/aml-quickstarts-267614/workspaces/quick-starts-ws-267614&tid=660b3398-b80e-49d2-bc5b-ac1dc93b5254

Streaming azureml-logs/hyperdrive.txt

[2024-09-15T09:29:19.643022][GENERATOR][INFO]Trying to sample '4' jobs from the hyperparameter space
[2024-09-15T09:29:20.2298504Z][SCHEDULER][INFO]Scheduling job, id='HD_d0709859-5831-49d2-b644-561bc8eb1e4d_0' 
[2024-09-15T09:29:20.4810531Z][SCHEDULER][INFO]Scheduling job, id='HD_d0709859-5831-49d2-b644-561bc8eb1e4d_2' 
[2024-09-15T09:29:20.3588792Z][SCHEDULER][INFO]Scheduling job, id='HD_d0709859-5831-49d2-b644-561bc8eb1e4d_1' 
[2024-09-15T09:29:20.6031279Z][SCHEDULER][INFO]Scheduling job, id='HD_d0709859-5831-49d2-b644-561bc8eb1e4d_3' 
[2024-09-15T09:29:20.568647][GENERATOR][INFO]Successfully sampled '4' jobs, they will soon be submitted to the execution t

{'runId': 'HD_d0709859-5831-49d2-b644-561bc8eb1e4d',
 'target': 'project-1',
 'status': 'Completed',
 'startTimeUtc': '2024-09-15T09:29:18.812034Z',
 'endTimeUtc': '2024-09-15T09:48:27.508641Z',
 'services': {},
 'properties': {'primary_metric_config': '{"name":"Accuracy","goal":"maximize"}',
  'resume_from': 'null',
  'runTemplate': 'HyperDrive',
  'azureml.runsource': 'hyperdrive',
  'platform': 'AML',
  'ContentSnapshotId': 'c8c68cfe-d914-4ec2-895f-e0bb403d8fa0',
  'user_agent': 'python/3.10.6 (Linux-4.15.0-1178-azure-x86_64-with-glibc2.27) msrest/0.6.21 Hyperdrive.Service/1.0.0 Hyperdrive.SDK/core.1.47.0',
  'space_size': 'infinite_space_size',
  'best_child_run_id': 'HD_d0709859-5831-49d2-b644-561bc8eb1e4d_10',
  'score': '0.9083459787556905',
  'best_metric_status': 'Succeeded',
  'best_data_container_id': 'dcid.HD_d0709859-5831-49d2-b644-561bc8eb1e4d_10'},
 'inputDatasets': [],
 'outputDatasets': [],
 'runDefinition': {'configuration': None,
  'attribution': None,
  'telemetryVa

In [7]:
import joblib
# Get your best run and save the model from that run.

### YOUR CODE HERE ###
best_run=exp_run.get_best_run_by_primary_metric()
print(best_run)

StatementMeta(144621e8-07d0-4231-8e1f-8c74d0028a6b, 0, 12, Finished, Available, Finished)

Run(Experiment: udacity-project,
Id: HD_d0709859-5831-49d2-b644-561bc8eb1e4d_10,
Type: azureml.scriptrun,
Status: Completed)


In [11]:
#print(best_run.get_details())
print(best_run.get_metrics())
print(best_run.get_file_names())

StatementMeta(6c912745-d107-4a12-86f9-8f44b6841ca6, 1, 16, Finished, Available, Finished)

{'Max iterations:': 64, 'Regularization Strength:': 1.0699733452615976, 'Accuracy': 0.9083459787556905}
['logs/azureml/dataprep/0/backgroundProcess.log', 'logs/azureml/dataprep/0/backgroundProcess_Telemetry.log', 'logs/azureml/dataprep/0/rslex.log.2024-09-03-10', 'system_logs/cs_capability/cs-capability.log', 'system_logs/hosttools_capability/hosttools-capability.log', 'system_logs/lifecycler/execution-wrapper.log', 'system_logs/lifecycler/lifecycler.log', 'system_logs/metrics_capability/metrics-capability.log', 'system_logs/snapshot_capability/snapshot-capability.log', 'user_logs/std_log.txt']


In [8]:
joblib.dump(best_run.get_metrics(),'best_run.json')

StatementMeta(144621e8-07d0-4231-8e1f-8c74d0028a6b, 0, 13, Finished, Available, Finished)

['best_run.json']

In [37]:
from azureml.data.dataset_factory import TabularDatasetFactory

# Create TabularDataset using TabularDatasetFactory
# Data is available at: 
# "https://automlsamplenotebookdata.blob.core.windows.net/automl-sample-notebook-data/bankmarketing_train.csv"

### YOUR CODE HERE ###
url = "https://automlsamplenotebookdata.blob.core.windows.net/automl-sample-notebook-data/bankmarketing_train.csv"
ds = TabularDatasetFactory.from_delimited_files(path=url)

StatementMeta(144621e8-07d0-4231-8e1f-8c74d0028a6b, 0, 42, Finished, Available, Finished)

In [40]:
!pip install -U pandas
!pip install --upgrade executing

StatementMeta(144621e8-07d0-4231-8e1f-8c74d0028a6b, 0, 45, Finished, Available, Finished)

In [27]:
!pip list | grep executing

StatementMeta(144621e8-07d0-4231-8e1f-8c74d0028a6b, 0, 32, Finished, Available, Finished)

executing                            1.0.0


In [28]:
!pip install -U ipython

StatementMeta(144621e8-07d0-4231-8e1f-8c74d0028a6b, 0, 33, Finished, Available, Finished)

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 819.0/819.0 kB 15.9 MB/s eta 0:00:00a 0:00:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 85.4/85.4 kB 20.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 386.4/386.4 kB 57.0 MB/s eta 0:00:00
  Using cached executing-2.1.0-py2.py3-none-any.whl (25 kB)
  Attempting uninstall: traitlets
    Found existing installation: traitlets 5.5.0
    Uninstalling traitlets-5.5.0:
      Successfully uninstalled traitlets-5.5.0
  Attempting uninstall: prompt-toolkit
    Found existing installation: prompt-toolkit 3.0.32
    Uninstalling prompt-toolkit-3.0.32:
      Successfully uninstalled prompt-toolkit-3.0.32
  Attempting uninstall: executing
    Found existing installation: executing 1.0.0
    Uninstalling executing-1.0.0:
      Successfully uninstalled executing-1.0.0
  Attempting uninstall: ipython
    Found existing installation: ipython 8.6.0
    Uninstalling ipython-8.6.0:
      Successfully uninstalled ipython-8.6.0
ERROR: pip's depe

In [64]:
!pip install -U azureml-sdk[notebooks,automl]

StatementMeta(144621e8-07d0-4231-8e1f-8c74d0028a6b, 0, 69, Finished, Available, Finished)

  Using cached azureml_dataprep-5.1.6-py3-none-any.whl (252 kB)
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 99.8/99.8 kB 3.9 MB/s eta 0:00:00
  Using cached pandas-1.3.5-cp310-cp310-manylinux_2_17_x86_64.manylinux2014_x86_64.whl (11.5 MB)
  Using cached azureml_dataprep_rslex-2.22.4-cp310-cp310-manylinux1_x86_64.whl (24.8 MB)
  Using cached azureml_dataprep_native-41.0.0-cp310-cp310-manylinux1_x86_64.whl (187 kB)
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 177.5/177.5 kB 12.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 157.9/157.9 kB 35.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 50.2 MB/s eta 0:00:0000:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.4/13.4 MB 69.2 MB/s eta 0:00:0000:0100:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 117.2/117.2 kB 19.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 61.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.5/4.5 MB 75.7 MB/s eta 0:00:00ta 0:0

In [62]:
!pip install azureml-dataprep==1.9.1


StatementMeta(144621e8-07d0-4231-8e1f-8c74d0028a6b, 0, 67, Finished, Available, Finished)

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.7/27.7 MB 12.5 MB/s eta 0:00:0000:0100:01
ERROR: Could not find a version that satisfies the requirement azureml-dataprep-native<15.0.0,>=14.2.1 (from azureml-dataprep) (from versions: 38.0.0, 41.0.0)
ERROR: No matching distribution found for azureml-dataprep-native<15.0.0,>=14.2.1


In [65]:
#from Users.odl_user_267614.train import clean_data

from sklearn.linear_model import LogisticRegression
import argparse
import os
import numpy as np
from sklearn.metrics import mean_squared_error
import joblib
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder
import pandas as pd
from azureml.core.run import Run
from azureml.data.dataset_factory import TabularDatasetFactory

def clean_data(data):
    # Dict for cleaning data
    months = {"jan":1, "feb":2, "mar":3, "apr":4, "may":5, "jun":6, "jul":7, "aug":8, "sep":9, "oct":10, "nov":11, "dec":12}
    weekdays = {"mon":1, "tue":2, "wed":3, "thu":4, "fri":5, "sat":6, "sun":7}

    # Clean and one hot encode data
    #x_df = pd.DataFrame(data)
    #x_df = x_df.dropna()
    print(data)
    x_df = data.to_pandas_dataframe().dropna()
    jobs = pd.get_dummies(x_df.job, prefix="job")
    x_df.drop("job", inplace=True, axis=1)
    x_df = x_df.join(jobs)
    x_df["marital"] = x_df.marital.apply(lambda s: 1 if s == "married" else 0)
    x_df["default"] = x_df.default.apply(lambda s: 1 if s == "yes" else 0)
    x_df["housing"] = x_df.housing.apply(lambda s: 1 if s == "yes" else 0)
    x_df["loan"] = x_df.loan.apply(lambda s: 1 if s == "yes" else 0)
    contact = pd.get_dummies(x_df.contact, prefix="contact")
    x_df.drop("contact", inplace=True, axis=1)
    x_df = x_df.join(contact)
    education = pd.get_dummies(x_df.education, prefix="education")
    x_df.drop("education", inplace=True, axis=1)
    x_df = x_df.join(education)
    x_df["month"] = x_df.month.map(months)
    x_df["day_of_week"] = x_df.day_of_week.map(weekdays)
    x_df["poutcome"] = x_df.poutcome.apply(lambda s: 1 if s == "success" else 0)

    y_df = x_df.pop("y").apply(lambda s: 1 if s == "yes" else 0)
    return x_df, y_df
# Use the clean_data function to clean your data.
x, y = clean_data(ds)

StatementMeta(144621e8-07d0-4231-8e1f-8c74d0028a6b, 0, 70, Finished, Available, Finished)

Unexpected exception formatting exception. Falling back to standard exception


Traceback (most recent call last):
  File "/home/trusted-service-user/cluster-env/env/lib/python3.10/site-packages/IPython/core/interactiveshell.py", line 3433, in run_code
    cell_name : str
  File "/tmp/ipykernel_7796/2257362377.py", line 45, in <module>
    x, y = clean_data(ds)
  File "/tmp/ipykernel_7796/2257362377.py", line 23, in clean_data
    print(data)
  File "/home/trusted-service-user/cluster-env/env/lib/python3.10/site-packages/azureml/data/_loggerfactory.py", line 132, in wrapper
  File "/home/trusted-service-user/cluster-env/env/lib/python3.10/site-packages/azureml/data/abstract_dataset.py", line 1043, in __str__
  File "/home/trusted-service-user/cluster-env/env/lib/python3.10/site-packages/azureml/data/_loggerfactory.py", line 132, in wrapper
  File "/home/trusted-service-user/cluster-env/env/lib/python3.10/site-packages/azureml/data/abstract_dataset.py", line 1054, in __repr__
  File "/home/trusted-service-user/cluster-env/env/lib/python3.10/site-packages/azureml/da

In [ ]:
from sklearn.model_selection import train_test_split
x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.2, random_state=42)
training_data = x_train
training_data['y']=y_train

In [ ]:
datastore = ws.get_default_datastore()
import pandas as pd
pd.DataFrame(training_data).to_csv("tmp/training_data.csv", index=False)

In [ ]:
from azureml.core import Dataset
datastore.upload(src_dir='tmp/', target_path='data/')
my_data = Dataset.Tabular.from_delimited_files(path=[(datastore, ('data/training_data.csv'))])

In [ ]:
from azureml.train.automl import AutoMLConfig

# Set parameters for AutoMLConfig
# NOTE: DO NOT CHANGE THE experiment_timeout_minutes PARAMETER OR YOUR INSTANCE WILL TIME OUT.
# If you wish to run the experiment longer, you will need to run this notebook in your own
# Azure tenant, which will incur personal costs.
automl_config = AutoMLConfig(
    experiment_timeout_minutes=30,
    task="classification",
    primary_metric="accuracy",
    training_data=training_data,
    label_column_name='y',
    n_cross_validations=4)

In [2]:
# Submit your automl run

### YOUR CODE HERE ###
from azureml.core.experiment import Experiment
experiment = Experiment(ws, "project-1-automl")
autml_run = experiment.submit(config=automl_config, show_output=True)
RunDetails(autml_run).show()

In [ ]:
# Retrieve and save your best automl model.

### YOUR CODE HERE ###
automl_best_run = autml_run.get_best_child()
print(automl_best_run)